In [ ]:
import pandas as pd
import numpy as np
from utils.feature_group import load_feature_groups

In [ ]:
FEATURE_GROUPS = load_feature_groups()
FEATURES = [col for group in FEATURE_GROUPS.values() for col in group]

In [ ]:
df = pd.read_csv('../data/clinical/preprocess.csv')

In [ ]:
df[FEATURES]

In [ ]:
df = df[df["FL_UDSD"] != 2]

In [ ]:
t = (
    df
    .sort_values("VISITYR")
    .groupby("PTID", sort=False)
    .first()
    .reset_index()
)

In [ ]:
t.to_csv("../data/clinical/preprocess_complete.csv")

In [ ]:
# df[['FL_UDSD', 'NACCETPR']].value_counts()

In [ ]:
non_mri_cols = [col for group, cols in FEATURE_GROUPS.items() if group != "MRI" for col in cols]
df_non_mri = df[non_mri_cols]

print(f"Unique patients: {df['PTID'].nunique()}")
print(f"Total visits:    {len(df)}")
df_non_mri

In [ ]:
CATEGORICAL = ["SEX", "APOE4S", "FL_UDSD", "NACCETPR"]

first_visit = (
    df[non_mri_cols]
    .sort_values("VISITYR")
    .groupby("PTID", sort=False)
    .first()
    .reset_index()
)

print(f"Patients at first visit: {len(first_visit)}\n")

cat_cols  = [c for c in CATEGORICAL if c in first_visit.columns]
cont_cols = [c for c in first_visit.columns if c not in cat_cols + ["PTID", "VISITYR", "EDUC", "NACCAGE"]]

print("=== Continuous features ===")
display(first_visit[cont_cols].describe())

In [ ]:
print("\n=== Categorical features ===")
for col in cat_cols:
    print(f"\n{col}:")
    display(first_visit[col].value_counts(dropna=False).to_frame("count"))

In [ ]:
cont_stats = first_visit[cont_cols].describe().T.map(lambda x: float(f"{x:.4g}"))
cont_stats.index.name = "feature"
cont_stats.to_csv("stats/continuous_stats.csv")

In [ ]:
import matplotlib.pyplot as plt

UDSD_LABELS = {
    1.0: "NC",
    3.0: "SCD",
    4.0: "EMCI",
    5.0: "LMCI",
    6.0: "Dementia",
}

udsd_classes = sorted(UDSD_LABELS.keys())
colors = ["#4caf50", "#2196f3", "#ffe600", "#f07b0d", "#ff0b0bff"]

ncols = 4
nrows = -(-len(cont_cols) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5 *2))
axes = axes.flatten()

for i, col in enumerate(cont_cols):
    ax = axes[i]
    groups = [first_visit.loc[first_visit["FL_UDSD"] == cls, col].dropna().values
              for cls in udsd_classes]
    bp = ax.boxplot(groups, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2),
                    flierprops=dict(marker="o", markersize=3, alpha=0.4),
                    widths=0.5)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    ax.set_title(col, fontsize=10)
    ax.set_xticks(range(1, len(udsd_classes) + 1))
    ax.set_xticklabels([UDSD_LABELS[c] for c in udsd_classes], fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Continuous features by FL_UDSD class (first visit per patient)", fontsize=13, y=1.01)
fig.tight_layout()
plt.savefig("stats/continuous_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for col in cont_cols:
    fig, ax = plt.subplots(figsize=(7, 10))
    groups = [first_visit.loc[first_visit["FL_UDSD"] == cls, col].dropna().values
              for cls in udsd_classes]
    bp = ax.boxplot(groups, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2),
                    flierprops=dict(marker="o", markersize=4, alpha=0.5),
                    widths=0.5)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    ax.set_title(col, fontsize=13)
    ax.set_xticks(range(1, len(udsd_classes) + 1))
    ax.set_xticklabels([UDSD_LABELS[c] for c in udsd_classes], fontsize=11)
    ax.set_ylabel(col, fontsize=10)
    fig.tight_layout()
    plt.savefig(f"stats/boxplot_{col}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
dict_df = pd.read_csv("../data/clinical/dictionary.csv")

# APOE4S was renamed from COMBINED_NE4S during preprocessing
DICT_NAME_MAP = {"APOE4S": "COMBINED_NE4S"}

def parse_value_labels(values_str):
    mapping = {}
    if pd.isna(values_str):
        return mapping
    for line in str(values_str).strip().split("\n"):
        if " = " in line:
            key, label = line.split(" = ", 1)
            try:
                mapping[float(key.strip())] = label.strip()
            except ValueError:
                pass
    return mapping

In [ ]:
value_labels = {}
for col in cat_cols:
    dict_name = DICT_NAME_MAP.get(col, col)
    row = dict_df[dict_df["Variable_name"] == dict_name]
    if not row.empty:
        value_labels[col] = parse_value_labels(row.iloc[0]["Values"])

cat_frames = []
for col in cat_cols:
    vc = first_visit[col].value_counts(dropna=False).reset_index()
    vc.columns = ["value", "count"]
    vc["label"] = vc["value"].map(value_labels.get(col, {})).fillna("NaN")
    vc.insert(0, "feature", col)
    cat_frames.append(vc)

cat_stats = pd.concat(cat_frames, ignore_index=True)[["feature", "value", "label", "count"]]
cat_stats.sort_values(by=["feature","value"]).to_csv("stats/categorical_stats.csv", index=False)

print("Saved categorical_stats.csv")
cat_stats.sort_values(by=["feature","value"])

In [ ]:
naccetpr = cat_stats[cat_stats["feature"] == "NACCETPR"].copy()
naccetpr["value"] = naccetpr["value"].apply(lambda v: 1.0 if v == 1.0 else 0.0)
naccetpr["label"] = naccetpr["value"].map({1.0: "Alzheimer's disease (AD)", 0.0: "Non-AD"})
naccetpr = naccetpr.groupby(["feature", "value", "label"], as_index=False)["count"].sum()

cat_stats_final = (
    pd.concat([cat_stats[cat_stats["feature"] != "NACCETPR"], naccetpr], ignore_index=True)
    .sort_values(["feature", "value"])
)

cat_stats_final.sort_values(by=["feature","value"]).to_csv("stats/categorical_stats_AD_nonAD.csv", index=False)
print("Saved categorical_stats_AD_nonAD.csv")
cat_stats_final.sort_values(by=["feature","value"]).reset_index(drop=True)

In [ ]:
compl